# #6 A-LOTO 2x2 — train di Colab (1 job per sesi)

Matrix (edit `CONFIG` di Sel 2, satu job per jalan):

| # | AUG_MODE | HOLDOUT_TAXONOMY | HF_REPO |
|---|----------|------------------|---------|
| 1 | none | banking77 | itsdevice/lfm-decs1-A-loto-bank |
| 2 | none | clinc150 | itsdevice/lfm-decs1-A-loto-clinc |
| 3 | full | banking77 | itsdevice/lfm-decs1-E-loto-bank |
| 4 | full | clinc150 | itsdevice/lfm-decs1-E-loto-clinc |

**VM mati tengah jalan?** Runtime baru → Run all lagi: checkpoint di Drive, otomatis resume. Cek progres via Sel 3 kapan saja.

In [ ]:
# Sel 1 — sekali per VM: Drive + deps + skrip train
from google.colab import drive
drive.mount("/content/drive")
!pip install -q unsloth datasets transformers huggingface_hub
!mkdir -p /content/drive/MyDrive/jev-loto6
%cd /content/drive/MyDrive/jev-loto6
!curl -sLO https://raw.githubusercontent.com/Braincore-RnD/Decision-Models-Paper/main/code/train_decision_models/unsloth_kaggle_dec.py
!ls -la unsloth_kaggle_dec.py

In [ ]:
# Sel 2 — EDIT per job, lalu Run. Aman diulang (resume, log append).
import os, subprocess
from getpass import getpass

CONFIG = dict(
    SEED="3407", EPOCHS="3", BATCH="4", GRAD_ACCUM="8",
    AUG_MODE="none", HOLDOUT_TAXONOMY="banking77",   # <-- EDIT
    HF_REPO="itsdevice/lfm-decs1-A-loto-bank",          # <-- EDIT
)
job = CONFIG["HF_REPO"].split("/")[-1]
work = f"/content/drive/MyDrive/jev-loto6/{job}"
!mkdir -p {work}

env = dict(os.environ, MAX_SEQ_LEN="4096", MIXTURE="1", N_ROWS="20400",
           EARLY_STOP="1", FORCE_NEW="0", **CONFIG)
env["HF_TOKEN"] = getpass("HF token (write): ")
log = open(f"{work}/train.log", "a")
p = subprocess.Popen(["python3", "/content/drive/MyDrive/jev-loto6/unsloth_kaggle_dec.py"],
                     cwd=work, stdout=log, stderr=subprocess.STDOUT,
                     env=env, start_new_session=True)
print(f"job={job} pid={p.pid} log={work}/train.log")

In [ ]:
# Sel 3 — monitor. Jalankan berkala. Selesai = ada 'run_meta' + push HF.
import glob, json
JOB = "lfm-decs1-A-loto-bank"  # <-- EDIT samakan dgn HF_REPO Sel 2
work = f"/content/drive/MyDrive/jev-loto6/{JOB}"
!echo '=== LOG (tail) ==='; tail -c 2000 {work}/train.log
!echo '=== GPU ==='; nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv,noheader
!echo '=== CKPT ==='; ls {work}/out-u/ 2>/dev/null
for m in glob.glob(f"{work}/out-u/run_meta.json"):
    d = json.load(open(m))
    print("CAL:", d.get("calibrate"), "\nINTEST:", d.get("internal_test"), "\nEXTEST:", d.get("external_test"))